## Part A : Bi-encoder retrieval
We will use a bi-encoder to retrieve semantically similar passages and inspect where it succeeds and where it fails.

## Part B : Cross-encoder scoring on the same candidates
We will use a cross-encoder on the same query-document pairs and observe how it captures finer-grained relevance.

In [ ]:
# !pip -q install -U sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder, util

## Load models

- **Bi-encoder** for efficient retrieval
- **Cross-encoder** for fine-grained pairwise scoring

In [ ]:
bi_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

## A Tiny hand-crafted corpus

In [ ]:
corpus = [
    {
        "id": "D1",
        "text": """To reset your password, open the login page, click Forgot Password, and follow the instructions sent to your email."""
    },
    {
        "id": "D2",
        "text": """To change your profile picture, open account settings and upload a new image."""
    },
    {
        "id": "D3",
        "text": """The YouTube mobile app includes settings for notifications, downloads, watch history, privacy, captions, playback controls, reminders, data saving, video quality, accessibility, and account management. Users often visit settings to control offline downloads, restrict mobile data use, manage recommendations, change caption appearance, review privacy options, and configure background playback. The app also provides controls for reminders to take breaks, bedtime reminders, and history-related behavior.
To access these settings, open the YouTube app and tap your profile picture. From there, you can enter the settings area and browse categories related to playback, general preferences, privacy, and data usage. Depending on device version and account state, some settings may appear in slightly different sections. For example, data-saving controls may be grouped separately from playback controls, while captions and accessibility options may appear under a different heading.
Some users open this menu to adjust video quality on Wi-Fi versus mobile data. Others use it to manage downloads, clear or pause watch history, review privacy settings, control recommendation behavior, or reduce interruptions from notifications. In addition, there are controls for playback behavior, including whether the app should continue automatically to another video after the current one ends.
To turn off autoplay in the YouTube mobile app, open YouTube, tap your profile picture, go to Settings, open Autoplay, and switch Autoplay off."""
    },
    {
        "id": "D4",
        "text": """The YouTube mobile app, to save data you can turn off automatic playback or adjust video quality when on mobile networks."""
    }
]

corpus_df = pd.DataFrame(corpus)
corpus_df

## Define queries


In [ ]:
queries = [
    {
        "query": "How do I reset my password?",
        "relevant_id": "D1"
    },
    {
        "query": "How do I turn off autoplay in the YouTube mobile app?",
        "relevant_id": "D3"
    }
]

queries_df = pd.DataFrame(queries)
queries_df

## Encode the corpus once for bi-encoder retrieval

We normalize embeddings so dot product behaves like cosine similarity.

In [ ]:
corpus_texts = [doc["text"] for doc in corpus]
corpus_ids = [doc["id"] for doc in corpus]

corpus_embeddings = bi_model.encode(
    corpus_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# Part A - Bi-encoder retrieval


In [ ]:
def bi_retrieve(query, top_k=5):
    query_emb = bi_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    scores = util.dot_score(query_emb, corpus_embeddings)[0].cpu().numpy()
    ranked_idx = np.argsort(-scores)[:top_k]

    results = []
    for idx in ranked_idx:
        results.append({
            "id": corpus[idx]["id"],
            "text": corpus[idx]["text"],
            "bi_score": float(scores[idx])
        })
    return results

In [ ]:
def show_bi_results(query_obj, top_k=5):
    query = query_obj["query"]
    relevant_id = query_obj["relevant_id"]

    print("=" * 110)
    print("QUERY:", query)
    print("EXPECTED RELEVANT DOC:", relevant_id)
    print("-" * 110)

    results = bi_retrieve(query, top_k=top_k)

    for rank, doc in enumerate(results, start=1):
        marker = " <-- EXPECTED" if doc["id"] == relevant_id else ""
        print(f"{rank}. {doc['id']} | bi_score={doc['bi_score']:.4f}{marker}")
        print(f"   {doc['text']}")

    return results

## Run bi-encoder retrieval

In [ ]:
bi_results_q1 = show_bi_results(queries[0], top_k=5)

In [ ]:
bi_results_q2 = show_bi_results(queries[1], top_k=5)

## Summary table for Part A

In [ ]:
lab1_rows = []

for q in queries:
    results = bi_retrieve(q["query"], top_k=5)
    lab1_rows.append({
        "query": q["query"],
        "expected_id": q["relevant_id"],
        "bi_top1": results[0]["id"],
        "bi_top1_correct": results[0]["id"] == q["relevant_id"]
    })

lab1_df = pd.DataFrame(lab1_rows)
lab1_df

# Part B : Cross-encoder scoring on the same candidates

In [ ]:
def cross_score(query, docs):
    pairs = [(query, doc["text"]) for doc in docs]
    scores = cross_model.predict(pairs)

    results = []
    for doc, score in zip(docs, scores):
        item = dict(doc)
        item["cross_score"] = float(score)
        results.append(item)
    return results

In [ ]:
def show_cross_on_bi_candidates(query_obj, top_k=5):
    query = query_obj["query"]
    relevant_id = query_obj["relevant_id"]

    bi_results = bi_retrieve(query, top_k=top_k)
    cross_results = cross_score(query, bi_results)
    cross_results = sorted(cross_results, key=lambda x: x["cross_score"], reverse=True)

    print("=" * 110)
    print("QUERY:", query)
    print("EXPECTED RELEVANT DOC:", relevant_id)
    print("-" * 110)

    print("\nCross-encoder scoring on the same candidates:")
    for rank, doc in enumerate(cross_results, start=1):
        marker = " <-- EXPECTED" if doc["id"] == relevant_id else ""
        print(f"{rank}. {doc['id']} | bi_score={doc['bi_score']:.4f} | cross_score={doc['cross_score']:.4f}{marker}")
        print(f"   {doc['text']}")

    return bi_results, cross_results

## Run cross-encoder scoring on the same candidate set


In [ ]:
_, cross_results_q1 = show_cross_on_bi_candidates(queries[0], top_k=5)

In [ ]:
_, cross_results_q2 = show_cross_on_bi_candidates(queries[1], top_k=5)

## Compare rank before vs after cross-encoder scoring

In [ ]:
def get_rank(results, relevant_id):
    for i, doc in enumerate(results, start=1):
        if doc["id"] == relevant_id:
            return i
    return None

comparison_rows = []

for q in queries:
    bi_results = bi_retrieve(q["query"], top_k=5)
    cross_results = cross_score(q["query"], bi_results)
    cross_results = sorted(cross_results, key=lambda x: x["cross_score"], reverse=True)

    comparison_rows.append({
        "query": q["query"],
        "relevant_id": q["relevant_id"],
        "bi_rank": get_rank(bi_results, q["relevant_id"]),
        "cross_rank": get_rank(cross_results, q["relevant_id"]),
        "bi_top1": bi_results[0]["id"],
        "cross_top1": cross_results[0]["id"]
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df